# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nandini3206/flyrank-ml-workspace/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## 1. My rule and its reason codes

**The rule, in plain words:** A page is worth reviewing for refresh if it is **stale**
(old content) **and** still **visible** (getting real search impressions). Stale-but-invisible
pages aren't worth reviewing yet; visible-but-fresh pages likely aren't declining for staleness
reasons.

**Reason code:** `stale_and_visible` (the only reason code this rule outputs — it either fires
or it doesn't).

**Action label:** `review_for_refresh`

**Two signals checked before encoding the rule:**
1. **Staleness (`content_age_days`)** — behind FlyRank's refresh flags.
2. **Visibility (`mar_impressions`)** — behind the "quick-win" logic (a page must be seen to be
   worth fixing).

In [2]:
import os, getpass, duckdb
import pandas as pd, numpy as np

HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if not HF_TOKEN:
    HF_TOKEN = getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

# Same March(feature) / April(label) window as w03_data_contract.
dataset = con.execute(f"""
    WITH mar_perf AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS mar_impressions,
               SUM(gsc_clicks) AS mar_clicks
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-03'
        GROUP BY 1, 2
    ),
    apr_perf AS (
        SELECT client_hash_id, content_hash_id,
               SUM(gsc_impressions) AS apr_impressions
        FROM {TABLES['fact_daily']}
        WHERE month = '2026-04'
        GROUP BY 1, 2
    )
    SELECT m.client_hash_id, m.content_hash_id, m.mar_impressions, m.mar_clicks,
           c.content_created_date,
           CASE WHEN a.apr_impressions IS NULL THEN 1
                WHEN a.apr_impressions < 0.8 * m.mar_impressions THEN 1
                ELSE 0 END AS is_declining
    FROM mar_perf m
    LEFT JOIN apr_perf a ON m.client_hash_id=a.client_hash_id AND m.content_hash_id=a.content_hash_id
    LEFT JOIN {TABLES['dim_content']} c ON m.content_hash_id=c.content_hash_id
    WHERE m.mar_impressions >= 100
""").df()

dataset["content_age_days"] = (pd.Timestamp("2026-03-31") - pd.to_datetime(dataset["content_created_date"])).dt.days
dataset = dataset.dropna(subset=["content_age_days"])
print(f"Rows: {len(dataset):,}  |  base decline rate: {dataset['is_declining'].mean():.3f}")

# --- Signal 1: staleness ---
dataset["stale_bucket"] = np.where(dataset["content_age_days"] >= 365, "old (>=365d)", "new (<365d)")
sig1 = dataset.groupby("stale_bucket").agg(n=("content_hash_id","count"), decline_rate=("is_declining","mean"))
print("\nSignal 1 — staleness:\n", sig1)

# --- Signal 2: visibility/volume ---
dataset["visible_bucket"] = np.where(dataset["mar_impressions"] >= 500, "visible (>=500)", "low-visibility (<500)")
sig2 = dataset.groupby("visible_bucket").agg(n=("content_hash_id","count"), decline_rate=("is_declining","mean"))
print("\nSignal 2 — visibility:\n", sig2)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 101,441  |  base decline rate: 0.517

Signal 1 — staleness:
                   n  decline_rate
stale_bucket                     
new (<365d)   87051      0.520074
old (>=365d)  14390      0.501668

Signal 2 — visibility:
                            n  decline_rate
visible_bucket                            
low-visibility (<500)  39517      0.543943
visible (>=500)        61924      0.500565


**Signal 1 verdict — staleness: FALSE.** Old content (≥365d, n=14,390) declines at 50.2%
vs. new content (<365d, n=87,051) at 52.0% — a 1.8-point gap, essentially no difference.
Staleness alone does not predict decline in this slice. A clearly negative result — this
saved the rule from leaning on a signal that isn't real.

**Signal 2 verdict — visibility: OPPOSITE.** Low-visibility pages (<500 impressions,
n=39,517) decline at 54.4%, vs. visible pages (≥500, n=61,924) at 50.1% — the *opposite*
of what a naive "bigger pages are more at risk" story would predict. The visibility gate
in the rule isn't there to predict decline — it's there to ensure a page has enough
traffic to be worth fixing at all.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [3]:
stale   = (dataset["content_age_days"] >= 365).astype(int)
visible = (dataset["mar_impressions"] >= 500).astype(int)
dataset["score"] = stale * visible * dataset["mar_impressions"]
dataset["reason_code"] = np.where((stale & visible) == 1, "stale_and_visible", "not_flagged")
dataset["action"] = np.where((stale & visible) == 1, "review_for_refresh", "no_action")

queue = dataset.sort_values("score", ascending=False)

os.makedirs("work/outputs", exist_ok=True)
queue[["client_hash_id","content_hash_id","mar_impressions","content_age_days",
       "score","reason_code","action"]].to_csv("work/outputs/baseline_action_score.csv", index=False)

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

for k in (20, 50):
    print(f"Precision@{k}: {precision_at_k(dataset['score'], dataset['is_declining'], k):.3f}  "
          f"(base rate: {dataset['is_declining'].mean():.3f})")

Precision@20: 0.200  (base rate: 0.517)
Precision@50: 0.280  (base rate: 0.517)


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [4]:
top20 = queue.head(20)[["content_hash_id","mar_impressions","content_age_days","score","reason_code","action","is_declining"]]
print(top20.to_string(index=False))

         content_hash_id  mar_impressions  content_age_days    score       reason_code             action  is_declining
content_eadb33b5df496f4a         617124.0               375 617124.0 stale_and_visible review_for_refresh             0
content_ec2e0346994fb5a5         245276.0               434 245276.0 stale_and_visible review_for_refresh             1
content_0e03de7680314cd5         221310.0               375 221310.0 stale_and_visible review_for_refresh             0
content_8d7d99f109e19aa2         203497.0               375 203497.0 stale_and_visible review_for_refresh             0
content_4ffe18112a5642e3         186983.0               375 186983.0 stale_and_visible review_for_refresh             0
content_471d9cabce329a66         164885.0               375 164885.0 stale_and_visible review_for_refresh             0
content_fd2117c2c6790e4b         151166.0               410 151166.0 stale_and_visible review_for_refresh             1
content_e241d6415ac9e534         142304.

**Top-20 review:** 16 of the top 20 ranked pages are NOT actually declining
(is_declining = 0) — only 4 hits (rows for content_ec2e0346994fb5a5, content_fd2117c2c6790e4b,
content_fec55986a1868d62, content_cd3d932d4e1c8db0).

**Why this happens:** the score is `stale * visible * mar_impressions` — since stale and
visible are just 0/1 gates, once a page passes both, the score IS its raw impression count.
The rule ends up ranking by "biggest page that happens to be old and visible," not by
"most likely to decline." Big evergreen pages with huge impression counts tend to be
*stable*, not declining — so the rule systematically promotes the wrong pages to the top.

**What would make each type of pick wrong:**
- High-impression, non-declining hits (rows 1, 3–6, 8–10, 12–13, 15–19): these are
  large, stable pages — wrong because size alone was mistaken for risk. A page being big
  says nothing about its trend.
- The 4 correct hits (declining=1) all sit lower in the ranking (positions 2, 7, 11, 20) —
  suggesting decline risk isn't concentrated at the very top of an impression-sorted list.

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
feature_cols = ["mar_impressions", "content_age_days"]
leaky_terms = {"apr_impressions", "is_declining", "trend_pct", "trend_direction"}
print("Leakage check — score inputs:", feature_cols)
print("Any leaked terms present?", bool(leaky_terms & set(feature_cols)))

# Weak picks: borderline threshold cases in the top 20
weak = top20[(top20["content_age_days"] < 400) | (top20["mar_impressions"] < 700)]
print("\nBorderline/weak picks in top 20:\n", weak)

Leakage check — score inputs: ['mar_impressions', 'content_age_days']
Any leaked terms present? False

Borderline/weak picks in top 20:
                 content_hash_id  mar_impressions  content_age_days     score  \
67600  content_eadb33b5df496f4a         617124.0               375  617124.0   
59622  content_0e03de7680314cd5         221310.0               375  221310.0   
64042  content_8d7d99f109e19aa2         203497.0               375  203497.0   
61809  content_4ffe18112a5642e3         186983.0               375  186983.0   
16058  content_471d9cabce329a66         164885.0               375  164885.0   
61955  content_545bb6cc7081ded3         122905.0               375  122905.0   
6717   content_b17c1d1cb0a346d6         115947.0               396  115947.0   
60261  content_21309e9a83c83653         103187.0               375  103187.0   
78325  content_e73024da2a848e26          98304.0               375   98304.0   

             reason_code              action  is_declining  
6

**Leakage check:** confirmed clean — `feature_cols = ['mar_impressions', 'content_age_days']`
contains none of `apr_impressions`, `is_declining`, `trend_pct`, or `trend_direction`. No
future-window or label-derived inputs anywhere in the rule.

**Weak picks:** 9 of the top 20 sit right at the `content_age_days ≈ 375` cluster — just
barely past the 365-day stale cutoff, not meaningfully "old" in any real sense. This
threshold-adjacency is a weak-rule symptom: the rule can't distinguish a page that's 375
days old from one that's 366 days old, even though the real risk driver (per Section 1/3)
isn't staleness at all — it's something the score doesn't currently capture. **Conclusion:**
this baseline is honestly beatable — that's the point. Week 5's model needs to beat
Precision@20 = 0.200 and Precision@50 = 0.280.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.